# 03 · Classical course baseline: MFCC + GMM-HMM

Runs on Kaggle **on CPU** (no GPU needed). **Settings → Internet: On.** No manual uploads.

`scripts/train_gmm_hmm.py` trains on the train split (flat start, Viterbi re-estimation, mixture
splitting), chooses the LM weight and insertion penalty on dev, and decodes test once at the end
(settings: `configs/train_gmm_hmm.yaml`). `TRIAL = True` uses 300 train clips and 20 dev/test clips.

Output: `/kaggle/working/results_gmm_hmm.zip` (predictions, `.meta.json`, training log).

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"
TRIAL = False
WORKERS = 4

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q -b $BRANCH $REPO_URL repo
%cd /kaggle/working/repo
!git log -1 --format='commit %H'
!pip install -q -r requirements/train.txt
!pip freeze | grep -iE '^(torch|numpy|pandas|pyarrow|jiwer|pyyaml)=='

In [ ]:
!python -m pytest -q tests/test_gmm_hmm.py

In [ ]:
!python scripts/unpack_audio.py

In [ ]:
import subprocess, sys
extra = ["--limit-train", "300", "--limit-eval", "20", "--out-dir", "/kaggle/working/trial"] if TRIAL else []
r = subprocess.run([sys.executable, "scripts/train_gmm_hmm.py", "--workers", str(WORKERS), *extra])
print("exit code", r.returncode)

In [ ]:
if TRIAL:
    !head -c 3000 /kaggle/working/trial/train_logs/gmm_hmm.json
else:
    !cp work/models/gmm_hmm/model.npz /kaggle/working/gmm_hmm_model.npz
    !zip -q -r /kaggle/working/results_gmm_hmm.zip results/predictions/gmm_hmm.* results/train_logs/gmm_hmm.json
    !ls -lh /kaggle/working